In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Phase 4 · Deterministic release tables

This phase reads the combined Phase 2 tables and saved V3 decision ledger, then rebuilds the species summary, accepted-evidence candidates, feature summaries, and Pathogenic Interest Index. It does not call PubMed or Jev. The index retains its three component counts and normalizes scores within organism group.

In [2]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.pipelines.phase4_release import build_phase4

CONFIG = load_yaml("configs/default.yaml")
GROUPS = selected_organism_groups(CONFIG)
PUBMED_ROOT = Path(CONFIG["paths"]["pubmed_release"])
LEDGER = Path(CONFIG["paths"]["decision_ledger"])
print("Current run selection:", GROUPS)
print("Phase 2 release:", PUBMED_ROOT)
print("Decision ledger:", LEDGER, "| exists:", LEDGER.exists())
print("Phase 4 output:", CONFIG["paths"]["release_root"])
for file in ("species_abstracts.parquet", "abstracts.parquet"):
    path=PUBMED_ROOT/file
    print(file, "exists:", path.exists(), "rows:", f"{len(pd.read_parquet(path)):,}" if path.exists() else "n/a")
print("\nImplementation called by this notebook:\n")
print(inspect.getsource(build_phase4))

Current run selection: ['fungi']
Phase 2 release: data/02_pubmed/release
Decision ledger: data/03_decisions/parsed/decision_ledger.parquet | exists: True
Phase 4 output: data/04_release


species_abstracts.parquet exists: True rows: 3,450,866


abstracts.parquet exists: True rows: 163,687

Implementation called by this notebook:

def build_phase4(config_path: str = "configs/default.yaml", *, decision_path: str | None = None) -> dict[str, int]:
    config = load_yaml(config_path)
    paths, decision = config["paths"], config["decisions"]
    sampling = config.get("pubmed", {}).get("sampling", {})
    selected_groups = set(selected_organism_groups(config))
    taxonomy_root = Path(paths["taxonomy_release"])
    pubmed_root = Path(paths["pubmed_release"])
    species = pd.read_parquet(taxonomy_root / "species.parquet")
    matches = pd.read_parquet(pubmed_root / "species_abstracts.parquet")
    abstracts = pd.read_parquet(pubmed_root / "abstracts.parquet")
    ledger_path = Path(decision_path or paths["decision_ledger"])
    ledger = pd.read_parquet(ledger_path) if ledger_path.exists() else pd.DataFrame()
    out = Path(paths["release_root"]); out.mkdir(parents=True, exist_ok=True)

    if not matches.empty:
        search_count

## Rebuild release tables

This is a local, deterministic transformation. It is safe to rerun after a decision checkpoint or after changing release thresholds.

In [3]:
RUN_PHASE = True
if RUN_PHASE:
    RELEASE_COUNTS = build_phase4()
    display(pd.DataFrame([RELEASE_COUNTS]))
else:
    print("Set RUN_PHASE=True to rebuild the release tables.")

,species,accepted_evidence,feature_decisions
0,142201,2674,510641
